# 05 — Final KPI Computation & Data Load Prep
**NST DVA Capstone 2 | B_G6_DIGITOMICS**

**Sector:** EdTech / Digital Behaviour Analytics  
**Dataset:** Student Digital Behaviour Data  
**Notebook Purpose:** Compute final aggregated metrics and derived features requested by the team. Prep the dataset for Tableau dashboards and final data load.

---
## Pipeline Steps:
1. Compute final KPIs (e.g., composite risk flags)
2. Handle any final renaming/formatting for visualization
3. Export to `data/processed/final_tableau_load.csv`


In [12]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded for Final Load Prep.")

Libraries loaded for Final Load Prep.


In [13]:
# Load the cleaned dataset
PROCESSED_PATH = '../data/processed/student_digital_behaviour_cleaned.csv'
df = pd.read_csv(PROCESSED_PATH)

print(f"Initial Shape: {df.shape}")

Initial Shape: (500000, 58)


## 1.5 Data Sampling for Analysis Efficiency
To ensure the Tableau dashboard remains highly performant and responsive during analysis without exceeding memory limitations or slowing down visualizations, we will downsample the data by selecting 100,000 random rows. Since the original dataset is extremely large (~500k rows), randomly sampling a substantial subset preserves the statistical distributions and overall patterns while making analytical operations significantly faster.

In [14]:
# Sample 100,000 random rows to optimize performance
SAMPLE_SIZE = 100000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    # Re-assign sequential student IDs to maintain consistency
    df["student_id"] = range(1, len(df) + 1)
    print(f"Data successfully sampled down to {SAMPLE_SIZE} rows.")
else:
    print("Data belongs within the size limit. No sampling required.")

print(f"Current working shape: {df.shape}")

Data successfully sampled down to 100000 rows.
Current working shape: (100000, 58)


## 1. Compute Final KPIs
We will create a specific `Total_Risk_Flag` or binary indicator for students in extreme danger of failing or severe mental health degradation so the dashboard can filter on "High Risk Students".

In [15]:
# Create High Risk Student Flag
# Condition: Academic Risk > 75th percentile OR Wellbeing < 25th percentile
academic_threshold = df['academic_risk_score'].quantile(0.75)
wellbeing_threshold = df['wellbeing_index'].quantile(0.25)

df['is_high_risk'] = np.where(
    (df['academic_risk_score'] >= academic_threshold) | 
    (df['wellbeing_index'] <= wellbeing_threshold), 1, 0)

# Create an overall Digital Engagement Ratio
# Ratio of non-educational digital time to total digital time
# non-educational = social_media + short_video + entertainment
df['non_edu_screen_time'] = df['social_media_hours'] + df['short_video_hours'] + df['entertainment_content_hours']
df['total_screen_time'] = df['non_edu_screen_time'] + df['education_content_hours'] + df['online_learning_hours']

# Avoid division by zero
df['idle_engagement_ratio'] = np.where(
    df['total_screen_time'] > 0, 
    df['non_edu_screen_time'] / df['total_screen_time'], 
    0
)

print("KPIs Computed: 'is_high_risk' & 'idle_engagement_ratio'")
print(df[['is_high_risk', 'idle_engagement_ratio']].describe())

KPIs Computed: 'is_high_risk' & 'idle_engagement_ratio'
       is_high_risk  idle_engagement_ratio
count      100000.0          100000.000000
mean            1.0               0.465672
std             0.0               0.117330
min             1.0               0.000000
25%             1.0               0.391332
50%             1.0               0.470813
75%             1.0               0.545487
max             1.0               0.858084


## 2. Formatting & Categorical Binning for Tableau
Tableau works best when age groups and score quadrants are pre-binned rather than calculated on the fly in the visualization tool.

In [16]:
# Age Binning
bins = [9, 15, 19, 23, 30]
labels = ['10-15', '16-19', '20-23', '24-30']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

# Rounded scores for cleaner tooltips
df['productivity_score_rounded'] = df['productivity_score'].round(1)
df['wellbeing_index_rounded'] = df['wellbeing_index'].round(1)

# Round all floating point columns to 2 decimals
float_cols = df.select_dtypes(include=['float64', 'float32']).columns
df[float_cols] = df[float_cols].round(2)

print("Categorical bins created for Tableau.")

Categorical bins created for Tableau.


## 3. Final Export
Saving to the appropriate processed folder so Tableau can connect directly to this file.

In [17]:
FINAL_EXPORT_PATH = '../data/processed/final_tableau_load.csv'

# Exporting
df.to_csv(FINAL_EXPORT_PATH, index=False)

print(f" Final load ready for Tableau at: {FINAL_EXPORT_PATH}")
print(f"Final Data Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

 Final load ready for Tableau at: ../data/processed/final_tableau_load.csv
Final Data Shape: 100,000 rows × 65 columns
